In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from utils import ASSETS_DIR




In [ ]:
df = pd.read_parquet(ASSETS_DIR / 'nacc74_cleaned.parquet')
df.describe()

In [ ]:


# 1. Separiamo la matrice delle Feature (X) dal Target (y)
X = df.drop(columns=['TARGET'])
X.drop(columns=['LBDEVAL'], inplace=True, errors='ignore')
y = df['TARGET']


classi_plot = ['Lewy Body', 'Alzheimer', 'Sani']



In [ ]:
df_plot = X.copy()
df_plot['TARGET'] = y

sns.set_theme()

plt.figure(figsize=(10, 6))
sns.histplot(
    data=df_plot,
    x='CRAFTDVR',
    hue='TARGET',
    stat='probability',
    multiple='fill',
    common_norm=False,
    palette='Set2',
    edgecolor='black',
    alpha=0.8,
    bins=20
)

plt.title('Frequenze relative di CRAFTDVR per classe TARGET', fontsize=14)
plt.xlabel('CRAFTDVR', fontsize=12)
plt.ylabel('Frequenza relativa', fontsize=12)
plt.legend(title='TARGET', labels=['Sani (0)', 'Alzheimer (1)', 'Lewy Body (2)'])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(50, 20))
g = sns.displot(
    data=df_plot,
    x='NACCAGE',
    hue='TARGET',
    palette='Set2',
    alpha=0.7,
    multiple='stack',
    edgecolor='black',
    bins=30,
    legend=False
)



plt.legend(title="Classi Target", labels=classi_plot)
plt.tight_layout()
plt.show()

In [ ]:
def multiplot(title, rows, cols, features: list, hue):
    fig, axes = plt.subplots(rows, cols, figsize=(20, 10))

    axes_flatten = axes.flatten()

    for i, feature in enumerate(features):
       
        sns.histplot(
                df,
                x=feature,
                palette='Set2',
                hue=hue,
                bins=50,
                multiple="stack",
                ax=axes_flatten[i],
                kde=True
            )
        axes_flatten[i].set_title(f"Distribuzione {feature}")
    
   

    axes_flatten[2].legend(title="Classi Tagert", labels=classi_plot)
    plt.tight_layout()
    plt.title(title)
    plt.show()

features = ["UDSBENTD", "TRAILA", "ANIMALS", "VEG", "MOCATRAI", "UDSVERTN"]

multiplot(title="pre traslation", rows=2, cols=3, features=features, hue="TARGET")

In [ ]:
import numpy as np
import pandas as pd

def translate_exception_codes(df, soglia_salto=20, step=3):
    """
    Trasla i codici di eccezione clinica NACC (995-998) appena sopra 
    il valore massimo reale della distribuzione, mantenendo intatta 
    la dimensionalità e proteggendo lo scaling.
    """
    df_translated = df.copy()
    colonne_numeriche = df_translated.select_dtypes(include=[np.number]).columns
    colonne_modificate = 0
    
    for col in colonne_numeriche:
        unique_vals = np.sort(df_translated[col].dropna().unique())
        
        if len(unique_vals) > 4:
            diffs = np.diff(unique_vals)
            max_diff = np.max(diffs)
            
            # Se troviamo l'abisso tra i dati reali e i codici d'errore
            if max_diff >= soglia_salto:
                gap_index = np.argmax(diffs)
                vero_massimo = unique_vals[gap_index]
                
                mask_special = df_translated[col] > vero_massimo
                
                if mask_special.sum() > 0:
                    ultime_cifre = df_translated.loc[mask_special, col] % 10
                    
                    # Traslazione: Li accodiamo subito dopo il massimo reale, 
                    # spaziati da uno "step" (es. +5, +10, +15) per tenerli distinti
                    df_translated.loc[mask_special & (ultime_cifre == 5), col] = vero_massimo + (step * 1) # Fisico
                    df_translated.loc[mask_special & (ultime_cifre == 6), col] = vero_massimo + (step * 2) # Cognitivo
                    df_translated.loc[mask_special & (ultime_cifre == 7), col] = vero_massimo + (step * 3) # Altro
                    df_translated.loc[mask_special & (ultime_cifre == 8), col] = vero_massimo + (step * 4) # Rifiuto
                    
                    colonne_modificate += 1
       

    print(f"Translazione completata: codici di errore compattati in {colonne_modificate} feature.")
    return df_translated

# Applichiamo la tua intuizione ai dataset
df = translate_exception_codes(df)


In [ ]:
multiplot(title="post traslation", rows=2, cols=3, features=features, hue="TARGET")

In [ ]:
sns.displot(
    df,
    x="MOCATRAI",
    hue="TARGET",
    palette="Set2",
    multiple="stack"

)

plt.tight_layout()
plt.legend(title="Classi Tagert", labels=classi_plot)
plt.show()

In [ ]:
sns.displot(
    df,
    x="DEP",
    hue="TARGET",
    palette="Set2",
    multiple="stack",
    bins=2

)
plt.tight_layout()
plt.show()

In [ ]:
sns.displot(
    df,
    x="MOCATOTS",
    hue="TARGET",
    palette="Set2",
    multiple="dodge",
    stat="density",
    common_norm=False,
    kde=True,
    legend=False

)
plt.legend(title="Classi Target", labels=classi_plot)
plt.tight_layout()
plt.show()

In [ ]:

from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(8, 6))


sns.pointplot(
    data=df,
    x='TARGET', 
    y='MOCATOTS', 
    errorbar=('ci', 95),
    n_boot=1000, 
    capsize=0.1, 
    linestyle="none",
    palette='Set2',
    hue='TARGET',
    ax=axes[0],
    legend=False
)

sns.pointplot(
    data=df,
    x='TARGET', 
    y='MOCATOTS',
    estimator=np.median, 
    errorbar=('ci', 95), 
    n_boot=1000, 
    capsize=0.1, 
    linestyle="none", 
    palette='Set2',
    hue='TARGET',
    ax=axes[1],
    legend=False,

)

axes[0].set_title("CI. al 95% della media (Bootstrap)")
axes[1].set_title("CI. al 95% della mediana (Bootstrap)")

colori = sns.color_palette('Set2', 3)

elementi_legenda = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colori[0], markersize=10, label='Sani'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colori[1], markersize=10, label='Alzheimer'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colori[2], markersize=10, label='Lewy Body')
]

fig.legend(
    handles=elementi_legenda, 
    title="Classe Target",
    loc='center right', 
    bbox_to_anchor=(1.15, 0.5), # Il valore 1.15 sull'asse X la spinge fuori dai grafici
    frameon=True
)
plt.xlabel("Classe Target")
plt.show()

In [ ]:
sns.pairplot(
    df.sub(features),
    hue="TARGET"
)

In [ ]:
from utils import ASSETS_DIR
print(f"Salvataggio del dataframe pulito in: {ASSETS_DIR / 'final_df.parquet'}")
if not ASSETS_DIR.exists():
    ASSETS_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(ASSETS_DIR / 'final_df.parquet', index=False)